# 천안시 격자 분석용 Colab 노트북 - Google Drive 버전

Google Drive의 `MyDrive/Cheonan` 폴더에 경계 파일을 넣고 실행합니다.

추천 파일:
- `cheonan_adm_dong_boundary_20250630.zip` : 천안시 31개 행정동만 들어 있는 작은 경계 ZIP

대체 가능:
- 원본 `BND_ADM_DONG_PG.zip`
- 또는 압축 해제한 `BND_ADM_DONG_PG.shp/.shx/.dbf/.prj/.cpg`
- 원본 전체 경계를 쓸 경우 `센서스 공간정보 지역 코드.xlsx`가 있으면 코드 매칭에 사용하고, 없으면 `34011`, `34012` prefix로 천안시를 필터링합니다.

In [ ]:
# 1. 패키지 설치
!pip -q install geopandas folium mapclassify openpyxl

In [ ]:
# 2. Google Drive 마운트 및 작업 폴더 설정
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/Cheonan')
OUT_DIR = DATA_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('작업 폴더:', DATA_DIR)
print('저장 폴더:', OUT_DIR)
print('폴더 파일:')
for p in sorted(DATA_DIR.glob('*')):
    print('-', p.name)

In [ ]:
# 3. 설정
GRID_M = 500              # 250, 500, 1000 등으로 변경 가능
KEEP_MODE = 'intersects'  # 'intersects': 경계와 닿는 모든 셀 / 'center': 중심점이 천안시 내부인 셀
CLIP_TO_CITY = True       # True면 격자를 천안시 경계로 잘라 분석/시각화

KOREA_CENTRAL_BELT_2010 = '+proj=tmerc +lat_0=38 +lon_0=127 +k=1 +x_0=200000 +y_0=600000 +ellps=GRS80 +units=m +no_defs'

In [ ]:
# 4. 경계 파일 읽기
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

small_zip = DATA_DIR / 'cheonan_adm_dong_boundary_20250630.zip'
full_zip = DATA_DIR / 'BND_ADM_DONG_PG.zip'
small_shp = DATA_DIR / 'cheonan_adm_dong_boundary_20250630.shp'
full_shp = DATA_DIR / 'BND_ADM_DONG_PG.shp'
code_xlsx_candidates = list(DATA_DIR.glob('*지역 코드*.xlsx'))
code_xlsx = code_xlsx_candidates[0] if code_xlsx_candidates else None

if small_zip.exists():
    print('천안시 전용 경계 ZIP 사용:', small_zip.name)
    cheonan_dong = gpd.read_file(f'zip://{small_zip}', encoding='cp949')
elif small_shp.exists():
    print('천안시 전용 SHP 사용:', small_shp.name)
    cheonan_dong = gpd.read_file(small_shp, encoding='cp949')
elif full_zip.exists() or full_shp.exists():
    source = f'zip://{full_zip}' if full_zip.exists() else full_shp
    print('원본 전국 행정동 경계 사용:', source)
    bnd = gpd.read_file(source, encoding='cp949')
    bnd['ADM_CD'] = bnd['ADM_CD'].astype(str).str.zfill(8)

    if code_xlsx is not None:
        print('지역 코드 파일 사용:', code_xlsx.name)
        code_df = pd.read_excel(code_xlsx, sheet_name='2025년 6월', header=1, dtype=str).fillna('')
        for col in ['시도코드', '시군구코드', '읍면동코드', '시군구명칭']:
            code_df[col] = code_df[col].astype(str).str.strip()
        cheonan_code_df = code_df[code_df['시군구명칭'].str.contains('천안시', na=False)].copy()
        cheonan_code_df['ADM_CD'] = (
            cheonan_code_df['시도코드'].str.zfill(2)
            + cheonan_code_df['시군구코드'].str.zfill(3)
            + cheonan_code_df['읍면동코드'].str.zfill(3)
        )
        cheonan_codes = set(cheonan_code_df['ADM_CD'])
        cheonan_dong = bnd[bnd['ADM_CD'].isin(cheonan_codes)].copy()
    else:
        print('지역 코드 파일 없음: ADM_CD prefix 34011/34012로 필터링')
        cheonan_dong = bnd[bnd['ADM_CD'].str.startswith(('34011', '34012'))].copy()
else:
    raise FileNotFoundError('Cheonan 폴더에 cheonan_adm_dong_boundary_20250630.zip 또는 BND_ADM_DONG_PG.zip을 넣어 주세요.')

if cheonan_dong.crs is None:
    cheonan_dong = cheonan_dong.set_crs(KOREA_CENTRAL_BELT_2010)

cheonan_dong['ADM_CD'] = cheonan_dong['ADM_CD'].astype(str).str.zfill(8)
cheonan_dong = cheonan_dong.sort_values('ADM_CD').reset_index(drop=True)

print('선택된 행정동 수:', len(cheonan_dong))
print('CRS:', cheonan_dong.crs)
display(cheonan_dong[[c for c in ['BASE_DATE', 'ADM_CD', 'ADM_NM'] if c in cheonan_dong.columns]])

In [ ]:
# 5. 천안시 경계 dissolve 및 격자 생성
city = cheonan_dong.dissolve().reset_index(drop=True)
city_geom = city.geometry.iloc[0]
minx, miny, maxx, maxy = city.total_bounds

x0 = math.floor(minx / GRID_M) * GRID_M
x1 = math.ceil(maxx / GRID_M) * GRID_M
y0 = math.floor(miny / GRID_M) * GRID_M
y1 = math.ceil(maxy / GRID_M) * GRID_M

records = []
for x in np.arange(x0, x1, GRID_M):
    for y in np.arange(y0, y1, GRID_M):
        records.append({
            'grid_id': f'G{len(records)+1:05d}',
            'grid_m': GRID_M,
            'geometry': box(x, y, x + GRID_M, y + GRID_M),
        })

grid_all = gpd.GeoDataFrame(records, crs=cheonan_dong.crs)

if KEEP_MODE == 'center':
    grid = grid_all[grid_all.geometry.centroid.within(city_geom)].copy()
else:
    grid = grid_all[grid_all.intersects(city_geom)].copy()

grid = grid.reset_index(drop=True)

if CLIP_TO_CITY:
    grid_for_analysis = gpd.overlay(grid, city[['geometry']], how='intersection', keep_geom_type=True)
    # overlay 후에도 원래 grid_id는 유지됩니다.
else:
    grid_for_analysis = grid.copy()

print('전체 후보 셀:', len(grid_all))
print(f'선택 셀({KEEP_MODE}):', len(grid))
print('분석/시각화 셀:', len(grid_for_analysis))
display(grid.drop(columns='geometry').head())

In [ ]:
# 6. 정적 지도 PNG 저장
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 12))
city.plot(ax=ax, color='#dcebd8', edgecolor='#26342b', linewidth=1.2)
grid_for_analysis.boundary.plot(ax=ax, color='#4869b1', linewidth=0.35, alpha=0.75)
cheonan_dong.boundary.plot(ax=ax, color='#26342b', linewidth=0.55, alpha=0.75)
ax.set_aspect('equal')
ax.set_axis_off()
ax.set_title(f'Cheonan-si {GRID_M}m grid', fontsize=16, pad=14)

png_path = OUT_DIR / f'cheonan_grid_{GRID_M}m.png'
plt.savefig(png_path, dpi=220, bbox_inches='tight', facecolor='white')
plt.show()
print('저장:', png_path)

In [ ]:
# 7. Folium 인터랙티브 지도 HTML 저장
import folium

city_wgs = city.to_crs(4326)
dong_wgs = cheonan_dong.to_crs(4326)
grid_wgs = grid_for_analysis.to_crs(4326)
center = city_wgs.geometry.iloc[0].centroid

m = folium.Map(location=[center.y, center.x], zoom_start=11, tiles='CartoDB positron')

folium.GeoJson(
    grid_wgs[['grid_id', 'grid_m', 'geometry']].to_json(),
    name=f'{GRID_M}m grid',
    style_function=lambda feature: {
        'fillColor': '#dcebd8',
        'color': '#4869b1',
        'weight': 0.6,
        'fillOpacity': 0.12,
    },
    tooltip=folium.GeoJsonTooltip(fields=['grid_id', 'grid_m']),
).add_to(m)

folium.GeoJson(
    dong_wgs[[c for c in ['ADM_CD', 'ADM_NM', 'geometry'] if c in dong_wgs.columns]].to_json(),
    name='행정동 경계',
    style_function=lambda feature: {
        'fillColor': 'transparent',
        'color': '#26342b',
        'weight': 1.4,
        'fillOpacity': 0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['ADM_CD', 'ADM_NM'] if 'ADM_NM' in dong_wgs.columns else ['ADM_CD']),
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
html_path = OUT_DIR / f'cheonan_grid_{GRID_M}m_map.html'
m.save(html_path)
print('저장:', html_path)
m

In [ ]:
# 8. 격자 분석용 파일 저장
# GeoPackage는 좌표계/속성 보존이 좋아서 이후 분석용으로 권장합니다.
gpkg_path = OUT_DIR / f'cheonan_grid_{GRID_M}m.gpkg'
csv_path = OUT_DIR / f'cheonan_grid_{GRID_M}m_centers.csv'

grid_save = grid_for_analysis.copy()
grid_save['center_x'] = grid_save.geometry.centroid.x
grid_save['center_y'] = grid_save.geometry.centroid.y
grid_save.to_file(gpkg_path, layer='grid', driver='GPKG')
grid_save.drop(columns='geometry').to_csv(csv_path, index=False, encoding='utf-8-sig')

print('저장:', gpkg_path)
print('저장:', csv_path)

In [ ]:
# 9. 나중에 병원/사업체 같은 점 데이터를 격자별로 집계하는 템플릿
# 예시 데이터 조건: CSV에 lon, lat, category 컬럼이 있다고 가정합니다.
# 실제 파일명이 생기면 POINT_CSV만 바꿔서 사용하세요.

# POINT_CSV = DATA_DIR / 'business_or_hospital_points.csv'
# point_df = pd.read_csv(POINT_CSV)
#
# points = gpd.GeoDataFrame(
#     point_df,
#     geometry=gpd.points_from_xy(point_df['lon'], point_df['lat']),
#     crs='EPSG:4326'
# ).to_crs(grid_for_analysis.crs)
#
# # 천안시 내부 점만 남기고, 각 점이 어느 grid_id 안에 있는지 붙입니다.
# points = gpd.sjoin(points, city[['geometry']], predicate='within', how='inner').drop(columns=['index_right'])
# joined = gpd.sjoin(points, grid_for_analysis[['grid_id', 'geometry']], predicate='within', how='left')
#
# # category별 개수 집계. category가 없으면 groupby('grid_id').size()만 쓰면 됩니다.
# counts = (
#     joined.groupby(['grid_id', 'category'])
#     .size()
#     .unstack(fill_value=0)
#     .reset_index()
# )
#
# grid_analysis = grid_for_analysis.merge(counts, on='grid_id', how='left').fillna(0)
# display(grid_analysis.head())